# genpcb Colab launcher

唯一的訓練啟動 notebook（兩個模型共用，靠 `MODEL_CONFIG` 切換）。
邏輯一律在 repo 的 script 裡，這裡只有環境準備與 `!python -m` 呼叫。

事前準備：
1. repo push 到 GitHub（private 可，搭配 Colab Secrets 的 `GH_TOKEN`）
2. Colab Secrets 設好 `HF_TOKEN`、`WANDB_API_KEY`
3. Runtime → A100

In [ ]:
# ── 參數：A/B 只改這一行 ──
MODEL_CONFIG = 'model_qwen35_9b'   # 或 'model_gemma4_12b'
STAGE = 'sft'                      # 'smoke_tokenizer' | 'sft' | 'grpo'
!nvidia-smi -L

In [ ]:
# ── Drive（checkpoint 落地點）與 secrets ──
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
CKPT_DIR = '/content/drive/MyDrive/genpcb/experiments'   # Colab 會斷線；checkpoint 只能放 Drive 或 HF Hub
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# ── 拉程式碼 + 安裝（repo 是 private，clone 帶 GH_TOKEN）──
REPO_AUTH = f"https://{userdata.get('GH_TOKEN')}@github.com/timmytsaa/genpcb.git"
import os.path
if os.path.isdir('/content/genpcb'):
    !git -C /content/genpcb pull
else:
    !git clone {REPO_AUTH} /content/genpcb
%pip install -q -e /content/genpcb
%pip install -q unsloth trl datasets wandb

In [ ]:
# ── 跑（所有邏輯在 repo script；這裡不寫訓練碼）──
# 資料與 checkpoint 都放 Drive（session 會斷）；DATA 由 prepare_data.ipynb 先產好
cfg = f'/content/genpcb/configs/{MODEL_CONFIG}.yaml'
DATA = '/content/drive/MyDrive/genpcb/data/sft/boards.jsonl'
OUT_ROOT = '/content/drive/MyDrive/genpcb/experiments'
if STAGE == 'smoke_tokenizer':
    !python -m genpcb.train.smoke_tokenizer --config {cfg} --out {CKPT_DIR}/smoke_tokenizer.json
elif STAGE == 'sft':
    !python -m genpcb.train.sft --config {cfg} --dataset {DATA} --output-root {OUT_ROOT} --resume
elif STAGE == 'grpo':
    !python -m genpcb.train.grpo --config {cfg} --output-root {OUT_ROOT} --resume

## 斷線恢復

Colab session 死掉後：重跑上面所有 cell（Drive 上的 checkpoint 還在），
sft/grpo script 將支援 `--resume` 自動從 `CKPT_DIR` 最新 checkpoint 續訓（Phase 2 實作）。
分析、畫圖**不要**在這本做——回本機開分析 notebook 讀 W&B / Drive artifacts。